In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pybedtools

In [128]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
HIMORNA_INITIAL = DATA_INITIAL / "himorna"
HIMORNA = DATA / "himorna"
HM_INITIAL = HIMORNA_INITIAL / "histone_modifications"
HM = HIMORNA / "histone_modifications"
METHYL_INITIAL = HIMORNA_INITIAL / "methylation"
METHYL = HIMORNA / "methylation"
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
HIMORNA_INDEX = PREPROCESSING / "himorna_index_new"
HIMORNA_GENES_SIGN =  PREPROCESSING / "himorna_genes_sign_new"
GENCODE = DATA / "gencode"

In [3]:
for directory in DATA, HIMORNA_INITIAL, HIMORNA:
    directory.mkdir(exist_ok=True, parents=True)

## Переаннотирование только уникальных пиков в pd.concat(hm_peaksm, methyl_peaks)

In [4]:
def get_peaks_coord(to_peaks):
    himorna_peaks = pd.read_csv(
            to_peaks,
            chunksize=100000000,
            sep="\t",
            usecols=["chrom", "start", "end"]
        )
    himorna_peaks = pd.concat(himorna_peaks)    
    himorna_peaks_coord = himorna_peaks[["chrom", "start", "end"]].drop_duplicates(ignore_index=True)
    return himorna_peaks_coord

In [5]:
def count_genome_coverage(anno):
    anno_bed_sort = pybedtools.BedTool.from_dataframe(anno).sort()
    genome_coverage = anno_bed_sort.genome_coverage(bga=True, genome='hg38')
    genome_coverage = genome_coverage.to_dataframe()
    genome_coverage["segment_lenght"] = genome_coverage["end"] - genome_coverage["start"]
    res = genome_coverage[genome_coverage["name"] != 0]["segment_lenght"].sum()
    return round(res / 3209286105 * 100, 2) # 3209286105 bp - размер генома человека

In [6]:
to_hm_peaks = HM_INITIAL / "Himorna_all.tsv.gz"
to_methyl_peaks = METHYL / "methylation_peaks.tsv.gz"

In [7]:
peaks_coord = {}
for table in [("hm_peaks",to_hm_peaks), ("methyl_peaks",to_methyl_peaks)]:
    print(table[0])
    peaks_coord[table[0]] = get_peaks_coord(to_peaks=table[1])

hm_peaks
methyl_peaks


In [8]:
[len(peaks_coord[i]) for i in peaks_coord]

[3968185, 27405966]

In [12]:
27405966 / 3968185 #Пиков метилирования больше в 7 раз, по сравнению с пиками гистоновых модификаций

6.906423465639833

In [9]:
himorna_peaks_coord = pd.concat([peaks_coord["hm_peaks"], peaks_coord["methyl_peaks"]], ignore_index=True)
himorna_peaks_coord.shape

(31374151, 3)

In [10]:
assert 3968185 + 27405966 == len(himorna_peaks_coord)

In [11]:
himorna_peaks_coord = himorna_peaks_coord.drop_duplicates(ignore_index=True)
himorna_peaks_coord.shape

(31374151, 3)

In [12]:
peaks_coord["hm_peaks"]["chrom"].nunique()

134

In [13]:
peaks_coord["methyl_peaks"]["chrom"].nunique()

25

In [14]:
himorna_peaks_coord["chrom"].nunique()

134

In [15]:
himorna_peaks_coord

,chrom,start,end
0,chr18,10441312,10441530
1,chr18,10441611,10441762
2,chr18,10442237,10443200
3,chr18,10443212,10443546
4,chr18,10443735,10444053
...,...,...,...
31374146,chrX,103029665,103029666
31374147,chrX,119678095,119678096
31374148,chrX,150307844,150307845
31374149,chr8,45570918,45570919


Так как для корректной работы pybedtools во время пересечения двух разметок необходимы лишь каноничные хромосомы, оставим только их

In [16]:
canonical_chrom = ["chr"+str(i) for i in range(1,23)]
canonical_chrom.append("chrX")
canonical_chrom.append("chrY")
canonical_chrom.append("chrM")
len(canonical_chrom)

25

In [17]:
mask = himorna_peaks_coord["chrom"].isin(canonical_chrom)
himorna_peaks_coord_canon_chrom = himorna_peaks_coord[mask]
himorna_peaks_coord_canon_chrom.shape

(31369304, 3)

31374151 - 31369304 = #Потеряли не так много пиков, ничего страшного

In [18]:
(31374151 - 31369304) / 31374151 * 100

0.01544902362457553

In [19]:
count_genome_coverage(anno=himorna_peaks_coord_canon_chrom)

79.73

In [20]:
himorna_peaks_coord_canon_chrom_sortbed = pybedtools.BedTool.from_dataframe(himorna_peaks_coord_canon_chrom).sort()

In [21]:
to_file = FANTOM / 'fantom_l2_genes-hg38-sort.bed'
fantom_genes_hg38 = pd.read_csv(to_file, sep="\t", header=None, names=["chrom","start","end","name"], usecols=[0,1,2,3])
fantom_genes_hg38.head(3)

,chrom,start,end,name
0,chr1,159537,162485,CATG00000042730
1,chr1,273882,274416,CATG00000042731
2,chr1,588120,634563,CATG00000042732


In [142]:
fantom_genes_hg38[fantom_genes_hg38["name"] == "<retired>"]

,chrom,start,end,name


In [22]:
len(fantom_genes_hg38)

121781

In [23]:
fantom_genes_hg38[fantom_genes_hg38["name"] == "CATG00000006156"]

,chrom,start,end,name
19476,chr11,74390265,74394560,CATG00000006156


In [24]:
fantom_genes_hg38_sortbed = pybedtools.BedTool.from_dataframe(fantom_genes_hg38).sort()

In [25]:
himorna_peaks_coord_assoc_sortbed = fantom_genes_hg38_sortbed.window(himorna_peaks_coord_canon_chrom_sortbed, w=1000).sort()

In [26]:
himorna_peaks_coord_assoc_sortbed.head(5)

chr1	273882	274416	CATG00000042731	chr1	273897	273898
 chr1	273882	274416	CATG00000042731	chr1	273896	273897
 chr1	588120	634563	CATG00000042732	chr1	612167	612168
 chr1	588120	634563	CATG00000042732	chr1	610710	610711
 chr1	588120	634563	CATG00000042732	chr1	612124	612125
 

In [27]:
%%time
len(himorna_peaks_coord_assoc_sortbed)

CPU times: user 4min 56s, sys: 9.23 s, total: 5min 5s
Wall time: 4min 57s


36907835

In [28]:
%%time
himorna_peaks_coord_assoc_sortbed.count()

CPU times: user 4min 59s, sys: 7.39 s, total: 5min 7s
Wall time: 4min 55s


36907835

In [29]:
himorna_peaks_coord_assoc_sortbed_v2 = himorna_peaks_coord_canon_chrom_sortbed.window(fantom_genes_hg38_sortbed, w=1000).sort()

In [30]:
himorna_peaks_coord_assoc_sortbed_v2.head(3)

chr1	273896	273897	chr1	273882	274416	CATG00000042731
 chr1	273897	273898	chr1	273882	274416	CATG00000042731
 chr1	587352	587353	chr1	588120	634563	CATG00000042732
 

In [31]:
himorna_peaks_coord_assoc_sortbed_v2.count()

36907835

Так числа совпали - .window работает корректно. Результат идентичен что при расширении пиков и поиске пересечений с генами, что при расширении генов и поиске пересечений с пиками

In [32]:
himorna_peaks_coord_assoc = pybedtools.BedTool.to_dataframe(himorna_peaks_coord_assoc_sortbed, disable_auto_names=True, names=["gene_chrom", "gene_start", "gene_end", "gene_ens", "chrom", "start", "end"])
himorna_peaks_coord_assoc.head(3)

,gene_chrom,gene_start,gene_end,gene_ens,chrom,start,end
0,chr1,273882,274416,CATG00000042731,chr1,273897,273898
1,chr1,273882,274416,CATG00000042731,chr1,273896,273897
2,chr1,588120,634563,CATG00000042732,chr1,612167,612168


In [33]:
len(himorna_peaks_coord_assoc)

36907835

Добавим gene_name:

In [83]:
to_file = FANTOM / 'fantom_l2-genes_with_name-hg38-all_info.tsv'
fantom_l2_genes_update_with_names_all_info = pd.read_csv(to_file, sep="\t", index_col=False)
fantom_l2_genes_update_with_names_all_info.head(3)

,chrom,start,end,ens_gene,score,strand,new-ens_gene,new-gene_name
0,chr1,159537,162485,CATG00000042730,.,-,CATG00000042730,CATG00000042730
1,chr1,273882,274416,CATG00000042731,.,-,CATG00000042731,CATG00000042731
2,chr1,588120,634563,CATG00000042732,.,-,CATG00000042732,CATG00000042732


In [88]:
fantom_l2_genes_update_with_names_all_info["new-ens_gene"].duplicated().sum()

2548

In [91]:
fantom_l2_genes_update_with_names_only_new = fantom_l2_genes_update_with_names_all_info[["new-ens_gene","new-gene_name"]].drop_duplicates().reset_index(drop=True)
fantom_l2_genes_update_with_names_only_new.head(3)

,new-ens_gene,new-gene_name
0,CATG00000042730,CATG00000042730
1,CATG00000042731,CATG00000042731
2,CATG00000042732,CATG00000042732


In [143]:
fantom_l2_genes_update_with_names_only_new[fantom_l2_genes_update_with_names_only_new["new-ens_gene"] == "<retired>"]

,new-ens_gene,new-gene_name
3,<retired>,<retired>


In [92]:
len(fantom_l2_genes_update_with_names_only_new)

121449

In [93]:
#Тут ошибки нет. gene_ens в левой таблице уже из обновленной разметки фантома
himorna_peaks_coord_assoc_with_gene_names = pd.merge(left=himorna_peaks_coord_assoc, right=fantom_l2_genes_update_with_names_only_new, how="left", left_on="gene_ens", right_on="new-ens_gene")
himorna_peaks_coord_assoc_with_gene_names.head(3)

,gene_chrom,gene_start,gene_end,gene_ens,chrom,start,end,new-ens_gene,new-gene_name
0,chr1,273882,274416,CATG00000042731,chr1,273897,273898,CATG00000042731,CATG00000042731
1,chr1,273882,274416,CATG00000042731,chr1,273896,273897,CATG00000042731,CATG00000042731
2,chr1,588120,634563,CATG00000042732,chr1,612167,612168,CATG00000042732,CATG00000042732


In [95]:
assert len(himorna_peaks_coord_assoc_with_gene_names) == len(himorna_peaks_coord_assoc)

In [98]:
columns = ["chrom","start","end","new-ens_gene","new-gene_name"]
himorna_peaks_coord_assoc_with_gene_names = himorna_peaks_coord_assoc_with_gene_names[columns]

In [102]:
himorna_peaks_coord_assoc_with_gene_names[himorna_peaks_coord_assoc_with_gene_names["start"] == 633422]

,chrom,start,end,new-ens_gene,new-gene_name
137,chr1,633422,633423,CATG00000042732,CATG00000042732
368,chr1,633422,633423,ENSG00000248527,MTATP6P1
521,chr1,633422,633423,ENSG00000198744,MTCO3P12
589,chr1,633422,633423,ENSG00000237973,MTCO1P12
643,chr1,633422,633423,ENSG00000229344,MTCO2P12
712,chr1,633422,633423,ENSG00000240409,MTATP8P1


In [103]:
himorna_peaks_coord_assoc_with_gene_names[(himorna_peaks_coord_assoc_with_gene_names["chrom"] == "chr11") & (himorna_peaks_coord_assoc_with_gene_names["start"] == "74366424") \
                        & (himorna_peaks_coord_assoc_with_gene_names["end"] == "74398020")]

,chrom,start,end,new-ens_gene,new-gene_name


In [104]:
himorna_peaks_coord_assoc_with_gene_names[(himorna_peaks_coord_assoc_with_gene_names["new-ens_gene"] == "CATG00000006156")]

,chrom,start,end,new-ens_gene,new-gene_name
5554055,chr11,74393641,74393642,CATG00000006156,CATG00000006156
5554056,chr11,74393606,74393607,CATG00000006156,CATG00000006156
5554057,chr11,74393464,74394037,CATG00000006156,CATG00000006156
5554058,chr11,74393369,74393370,CATG00000006156,CATG00000006156
5554059,chr11,74393368,74393369,CATG00000006156,CATG00000006156
...,...,...,...,...,...
5554134,chr11,74390250,74391630,CATG00000006156,CATG00000006156
5554135,chr11,74390624,74390625,CATG00000006156,CATG00000006156
5554136,chr11,74390625,74390626,CATG00000006156,CATG00000006156
5554137,chr11,74390684,74399166,CATG00000006156,CATG00000006156


## Загрузка, конкатенация и аннотация оригинальных таблиц

In [37]:
to_hm_peaks = HM_INITIAL / "Himorna_all.tsv.gz"
to_methyl_peaks = METHYL / "methylation_peaks.tsv.gz"

In [58]:
hm_peaks = pd.read_csv(
        to_hm_peaks,
        chunksize=100000000,
        sep="\t",
        index_col=False
    )
hm_peaks = pd.concat(hm_peaks)
print("hm_peaks: done")

hm_peaks: done


In [59]:
hm_peaks.shape

(104264744, 11)

In [146]:
mask = hm_peaks["lnc_ens"].apply(lambda x: "_" in x)
hm_peaks[mask]

,Unnamed: 0,hm,lnc,lnc_ens,peak,chrom,start,end,gene,gene_ens,corr


In [149]:
mask = hm_peaks["lnc_ens"].apply(lambda x: len(x) != len("ENSG00000259438"))
hm_peaks[mask]

,Unnamed: 0,hm,lnc,lnc_ens,peak,chrom,start,end,gene,gene_ens,corr


In [151]:
mask = hm_peaks["gene_ens"].apply(lambda x: len(x) != len("ENSG00000259438") if type(x) != float else False)
hm_peaks[mask]

,Unnamed: 0,hm,lnc,lnc_ens,peak,chrom,start,end,gene,gene_ens,corr


In [60]:
hm_peaks[(hm_peaks["lnc"] == "MAPK6-DT") & (hm_peaks["hm"] == "H4K20me1") & (hm_peaks["peak"] == "peak_38549") & (hm_peaks["chrom"] == "chr11")]

,Unnamed: 0,hm,lnc,lnc_ens,peak,chrom,start,end,gene,gene_ens,corr
104022425,104022425,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,LOC100287896,ENSG00000254837,-0.891363
104022426,104022426,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,MIR548AL,ENSG00000264402,-0.891363
104022427,104022427,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,NaN,CATG00000006156,-0.891363
104022428,104022428,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,AP001372.1,ENSG00000254631,-0.891363
104022429,104022429,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,PGM2L1,ENSG00000165434,-0.891363
104022430,104022430,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,NaN,CATG00000002919,-0.891363


In [61]:
columns = ["hm","lnc","lnc_ens","peak","chrom","start","end","corr"]

In [62]:
hm_peaks_without_assoc = hm_peaks[columns]
hm_peaks_without_assoc

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511
3,H3K27ac,AGAP11,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645
4,H3K27ac,AC015961.2,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911
...,...,...,...,...,...,...,...,...
104264739,H4K20me1,MAPK6-DT,ENSG00000259438,peak_18626,chr1,222815712,222856635,-0.730043
104264740,H4K20me1,AL158825.2,ENSG00000229587,peak_18626,chr1,222815712,222856635,-0.755635
104264741,H4K20me1,AL158825.2,ENSG00000229587,peak_18626,chr1,222815712,222856635,-0.755635
104264742,H4K20me1,AL158825.2,ENSG00000229587,peak_18626,chr1,222815712,222856635,-0.755635


In [63]:
hm_peaks_without_assoc = hm_peaks_without_assoc.drop_duplicates()
len(hm_peaks_without_assoc)

67288977

In [64]:
hm_peaks_without_assoc.head(3)

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511


In [65]:
hm_peaks_without_assoc[(hm_peaks_without_assoc["lnc"] == "MAPK6-DT") & (hm_peaks_without_assoc["hm"] == "H4K20me1") & (hm_peaks_without_assoc["peak"] == "peak_38549") & (hm_peaks_without_assoc["chrom"] == "chr11")]

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
104022425,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363


In [66]:
hm_peaks_without_assoc["lnc_ens"] = hm_peaks_without_assoc["lnc_ens"].astype('str')

/tmp/ipykernel_134492/3468657283.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hm_peaks_without_assoc["lnc_ens"] = hm_peaks_without_assoc["lnc_ens"].astype('str')


Вот тут отбирать пики только с какнонических хромосом НЕ НУЖНО. В коде мы смотрим чисто для справки, сколько пиков каждого класса

In [67]:
mask = hm_peaks_without_assoc["chrom"].isin(canonical_chrom)
hm_peaks_canon_chrom = hm_peaks_without_assoc[mask]
hm_peaks_canon_chrom.head(3)

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511


In [68]:
hm_peaks_canon_chrom.shape

(67014817, 8)

In [69]:
mask = hm_peaks_without_assoc["chrom"].isin(canonical_chrom)
hm_peaks_noncanon_chrom = hm_peaks_without_assoc[~mask]
hm_peaks_noncanon_chrom.head(3)

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
3891898,H3K27ac,AC053503.5,ENSG00000280323,peak_180459,chr11_KI270721v1_random,21849,22342,0.584858
3891899,H3K27ac,AC126696.3,ENSG00000260498,peak_180459,chr11_KI270721v1_random,21849,22342,-0.490055
3891900,H3K27ac,AC022336.2,ENSG00000260391,peak_180459,chr11_KI270721v1_random,21849,22342,0.477395


In [70]:
hm_peaks_noncanon_chrom.shape

(274160, 8)

In [40]:
methyl_peaks = pd.read_csv(
        to_methyl_peaks,
        chunksize=100000000,
        sep="\t",
    )
methyl_peaks = pd.concat(methyl_peaks)
print("methyl_peaks: done")

methyl_peaks: done


In [41]:
methyl_peaks.shape

(452430340, 7)

In [47]:
methyl_peaks

,chrom,start,end,peak,corr,lnc_ens,corr_sign
0,chr12,49123587,49123588,peak_10386212,-0.888,ENSG00000099869,minus
1,chr8,142304068,142304069,peak_49210615,0.911,ENSG00000099869,plus
2,chr1,15837335,15837336,peak_558140,-0.870,ENSG00000103472,minus
3,chr1,66118486,66118487,peak_1734200,0.864,ENSG00000103472,plus
4,chr1,199066625,199066626,peak_3431110,0.878,ENSG00000103472,plus
...,...,...,...,...,...,...,...
452430335,chr17,81669387,81669388,peak_21156386,-0.897,ENSG00000261222,minus
452430336,chr2,222626043,222626044,peak_28291839,-0.894,ENSG00000261222,minus
452430337,chr3,120774402,120774403,peak_33936371,-0.938,ENSG00000261222,minus
452430338,chr8,9741473,9741474,peak_47069123,-0.890,ENSG00000261222,minus


In [48]:
methyl_peaks["lnc_ens"].nunique()

1705

In [49]:
# Для метилирования не отбираем по хромосомам, поскольку там уже сожержатся только канонические

hm_peaks_canon_chrom["corr"] = np.round(hm_peaks_canon_chrom["corr"], 3)

In [50]:
methyl_peaks["lnc_ens"] = methyl_peaks["lnc_ens"].astype('str')

In [51]:
methyl_peaks["hm"] = "methylation"

In [52]:
columns = ["hm","lnc_ens","peak","chrom","start","end","corr"]

In [53]:
methyl_peaks = methyl_peaks[columns]

In [54]:
methyl_peaks = methyl_peaks.drop_duplicates()
len(methyl_peaks)

452430340

In [71]:
hm_peaks_without_assoc

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511
3,H3K27ac,AGAP11,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645
4,H3K27ac,AC015961.2,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911
...,...,...,...,...,...,...,...,...
104264727,H4K20me1,AC073655.2,ENSG00000258365,peak_18625,chr1,222804573,222806159,0.859463
104264728,H4K20me1,AC001226.1,ENSG00000274898,peak_18626,chr1,222815712,222856635,-0.833032
104264732,H4K20me1,AC004148.1,ENSG00000263272,peak_18626,chr1,222815712,222856635,-0.670470
104264736,H4K20me1,MAPK6-DT,ENSG00000259438,peak_18626,chr1,222815712,222856635,-0.730043


In [72]:
methyl_peaks

,hm,lnc_ens,peak,chrom,start,end,corr
0,methylation,ENSG00000099869,peak_10386212,chr12,49123587,49123588,-0.888
1,methylation,ENSG00000099869,peak_49210615,chr8,142304068,142304069,0.911
2,methylation,ENSG00000103472,peak_558140,chr1,15837335,15837336,-0.870
3,methylation,ENSG00000103472,peak_1734200,chr1,66118486,66118487,0.864
4,methylation,ENSG00000103472,peak_3431110,chr1,199066625,199066626,0.878
...,...,...,...,...,...,...,...
452430335,methylation,ENSG00000261222,peak_21156386,chr17,81669387,81669388,-0.897
452430336,methylation,ENSG00000261222,peak_28291839,chr2,222626043,222626044,-0.894
452430337,methylation,ENSG00000261222,peak_33936371,chr3,120774402,120774403,-0.938
452430338,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890


In [73]:
himorna_peaks = pd.concat([hm_peaks[["hm","lnc_ens","peak","chrom","start","end","corr"]], methyl_peaks], ignore_index=True)

In [77]:
himorna_peaks = himorna_peaks.reset_index(drop=True)

In [108]:
himorna_peaks

,hm,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356
1,H3K27ac,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105
2,H3K27ac,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511
3,H3K27ac,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645
4,H3K27ac,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911
...,...,...,...,...,...,...,...
556695079,methylation,ENSG00000261222,peak_21156386,chr17,81669387,81669388,-0.897000
556695080,methylation,ENSG00000261222,peak_28291839,chr2,222626043,222626044,-0.894000
556695081,methylation,ENSG00000261222,peak_33936371,chr3,120774402,120774403,-0.938000
556695082,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000


In [80]:
himorna_peaks.shape

(556695084, 7)

In [105]:
himorna_peaks_assoc = pd.merge(left=himorna_peaks, right=himorna_peaks_coord_assoc_with_gene_names, how="left", left_on=["chrom", "start", "end"], right_on=["chrom", "start", "end"])

In [106]:
himorna_peaks_assoc

,hm,lnc_ens,peak,chrom,start,end,corr,new-ens_gene,new-gene_name
0,H3K27ac,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN
1,H3K27ac,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN
2,H3K27ac,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN
3,H3K27ac,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645,NaN,NaN
4,H3K27ac,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911,NaN,NaN
...,...,...,...,...,...,...,...,...,...
836838146,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,ENSG00000173273,TNKS
836838147,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,ENSG00000207701,MIR597
836838148,methylation,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,ENSG00000171056,SOX7
836838149,methylation,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,ENSG00000258724,PINX1


Размер таблицы увеличился, так как один и тот же пик может быть аннотирован несколькими генами:

In [119]:
himorna_peaks[(himorna_peaks["lnc_ens"] == "ENSG00000261222") & (himorna_peaks["start"] == 9741473)]

,hm,lnc_ens,peak,chrom,start,end,corr
556695082,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.89


In [118]:
himorna_peaks_assoc[(himorna_peaks_assoc["lnc_ens"] == "ENSG00000261222") & (himorna_peaks_assoc["start"] == 9741473)]

,hm,lnc_ens,peak,chrom,start,end,corr,new-ens_gene,new-gene_name
836838146,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.89,ENSG00000173273,TNKS
836838147,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.89,ENSG00000207701,MIR597


Обновим lnc_ens и добавим gene_name

In [130]:
to_file = GENCODE / 'gencode_v31-to-v47-genes-hg38-with_names-all_info.tsv'
gencode_v31_to_v47_genes_hg38_with_names = pd.read_csv(to_file, sep="\t", index_col=False)
gencode_v31_to_v47_genes_hg38_with_names

,chrom,start,end,score,strand,ens_gene,gene_name,new-ens_gene,new-gene_name
0,chr1,11869,14409,.,+,ENSG00000223972,DDX11L1,ENSG00000223972,DDX11L1
1,chr1,14404,29570,.,-,ENSG00000227232,WASH7P,ENSG00000227232,WASH7P
2,chr1,17369,17436,.,-,ENSG00000278267,MIR6859-1,ENSG00000278267,MIR6859-1
3,chr1,29554,31109,.,+,ENSG00000243485,MIR1302-2HG,ENSG00000243485,MIR1302-2HG
4,chr1,30366,30503,.,+,ENSG00000284332,MIR1302-2,ENSG00000284332,MIR1302-2
...,...,...,...,...,...,...,...,...,...
60598,chrM,14149,14673,.,-,ENSG00000198695,MT-ND6,ENSG00000198695,MT-ND6
60599,chrM,14674,14742,.,-,ENSG00000210194,MT-TE,ENSG00000210194,MT-TE
60600,chrM,14747,15887,.,+,ENSG00000198727,MT-CYB,ENSG00000198727,MT-CYB
60601,chrM,15888,15953,.,+,ENSG00000210195,MT-TT,ENSG00000210195,MT-TT


In [134]:
himorna_peaks_assoc

,hm,lnc_ens,peak,chrom,start,end,corr,new-ens_gene,new-gene_name
0,H3K27ac,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN
1,H3K27ac,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN
2,H3K27ac,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN
3,H3K27ac,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645,NaN,NaN
4,H3K27ac,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911,NaN,NaN
...,...,...,...,...,...,...,...,...,...
836838146,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,ENSG00000173273,TNKS
836838147,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,ENSG00000207701,MIR597
836838148,methylation,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,ENSG00000171056,SOX7
836838149,methylation,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,ENSG00000258724,PINX1


In [132]:
himorna_peaks_assoc_with_names = pd.merge(left=himorna_peaks_assoc, right=gencode_v31_to_v47_genes_hg38_with_names[["ens_gene", "gene_name", "new-ens_gene","new-gene_name"]], how="left", left_on="lnc_ens", right_on="ens_gene")
himorna_peaks_assoc_with_names.head(3)

IOStream.flush timed out


,hm,lnc_ens,peak,chrom,start,end,corr,new-ens_gene_x,new-gene_name_x,ens_gene,gene_name,new-ens_gene_y,new-gene_name_y
0,H3K27ac,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN,ENSG00000279885,AP005060.1,ENSG00000279885,ENSG00000279885
1,H3K27ac,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN,ENSG00000277135,AC012409.3,ENSG00000277135,ENSG00000277135
2,H3K27ac,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN,ENSG00000273133,AC116651.1,ENSG00000273133,ENSG00000273133


In [133]:
len(himorna_peaks_assoc_with_names)

836906504

In [135]:
himorna_peaks_assoc_with_names = himorna_peaks_assoc_with_names.rename(columns={"new-ens_gene_x":"new-ens_gene", "new-gene_name_x":"new-gene_name",
                                               "gene_name":"lnc", "new-ens_gene_y":"new-lnc_ens", "new-gene_name_y":"new-lnc"})

IOStream.flush timed out
IOStream.flush timed out


In [136]:
del himorna_peaks_assoc_with_names["ens_gene"]

In [138]:
columns = ["hm","lnc","lnc_ens","new-lnc","new-lnc_ens","peak","chrom","start","end","corr","new-gene_name","new-ens_gene"]
himorna_peaks_assoc_with_names = himorna_peaks_assoc_with_names[columns]
himorna_peaks_assoc_with_names

,hm,lnc,lnc_ens,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene
0,H3K27ac,AP005060.1,ENSG00000279885,ENSG00000279885,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN
1,H3K27ac,AC012409.3,ENSG00000277135,ENSG00000277135,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN
2,H3K27ac,AC116651.1,ENSG00000273133,ENSG00000273133,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN
3,H3K27ac,AGAP11,ENSG00000271880,ENSG00000151303,ENSG00000151303,peak_389211,chr18,10441312,10441530,0.491645,NaN,NaN
4,H3K27ac,AC015961.2,ENSG00000267707,ENSG00000267707,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
836906499,methylation,AC064805.1,ENSG00000261222,CD300LD-AS1,ENSG00000182352,peak_47069123,chr8,9741473,9741474,-0.890000,TNKS,ENSG00000173273
836906500,methylation,AC064805.1,ENSG00000261222,CD300LD-AS1,ENSG00000182352,peak_47069123,chr8,9741473,9741474,-0.890000,MIR597,ENSG00000207701
836906501,methylation,AC064805.1,ENSG00000261222,CD300LD-AS1,ENSG00000182352,peak_47097647,chr8,10827627,10827628,-0.879000,SOX7,ENSG00000171056
836906502,methylation,AC064805.1,ENSG00000261222,CD300LD-AS1,ENSG00000182352,peak_47097647,chr8,10827627,10827628,-0.879000,PINX1,ENSG00000258724


In [139]:
to_save = HIMORNA / "himorna_peaks-hm_meth-assoc_with_names.tsv.gz"
himorna_peaks_assoc_with_names.to_csv(to_save, sep="\t", compression="gzip", index=False)

In [157]:
mask = (himorna_peaks_assoc_with_names["hm"] == "H3K9me3") & (himorna_peaks_assoc_with_names["lnc_ens"] == "ENSG00000260871") & (himorna_peaks_assoc_with_names["corr"] < 0)
H3K9me3_ENSG00000260871_minus = himorna_peaks_assoc_with_names[mask]

In [158]:
H3K9me3_ENSG00000260871_minus

,hm,lnc,lnc_ens,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene
241122454,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_39006,chr10,105890380,105891036,-0.396452,CATG00000000711,CATG00000000711
241123369,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_95489,chr14,90702013,90702723,-0.419884,TTC7B,ENSG00000165914
241123370,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_95489,chr14,90702013,90702723,-0.419884,CATG00000021841,CATG00000021841
241123371,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_95489,chr14,90702013,90702723,-0.419884,TTC7B,ENSG00000165914
241123372,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_95489,chr14,90702013,90702723,-0.419884,CATG00000021841,CATG00000021841
...,...,...,...,...,...,...,...,...,...,...,...,...
245542296,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_316352,chr8,132005903,132007751,-0.443203,CATG00000100998,CATG00000100998
245542302,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_316353,chr8,132011226,132011611,-0.529388,EFR3A,ENSG00000132294
245542764,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_316445,chr8,132590961,132591210,-0.405973,DNAAF11,ENSG00000129295
245542886,H3K9me3,AC093510.2,ENSG00000260871,ENSG00000260871,ENSG00000260871,peak_316458,chr8,132681446,132681813,-0.381626,NaN,NaN


In [28]:
to_file =  HIMORNA / "himorna_peaks-hm_meth-assoc.tsv.gz"
himorna_peaks_assoc = pd.read_csv(to_file, sep="\t")

In [122]:
himorna_peaks_assoc

,hm,lnc,lnc_ens,peak,chrom,start,end,corr,gene_chrom,gene_start,gene_end,gene_ens
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN,NaN,NaN
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN,NaN,NaN
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN,NaN,NaN
3,H3K27ac,AGAP11,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645,NaN,NaN,NaN,NaN
4,H3K27ac,AC015961.2,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
696878886,methylation,NaN,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,chr8,9551766.0,9788424.0,ENSG00000173273-gene
696878887,methylation,NaN,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,chr8,9740848.0,9741768.0,ENSG00000207701-gene
696878888,methylation,NaN,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,chr8,10712751.0,10839868.0,ENSG00000171056-gene
696878889,methylation,NaN,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,chr8,10725399.0,10839868.0,ENSG00000258724-gene


In [123]:
himorna_peaks_assoc[(himorna_peaks_assoc["lnc"] == "MAPK6-DT") & (himorna_peaks_assoc["hm"] == "H4K20me1") & (himorna_peaks_assoc["peak"] == "peak_38549") & (himorna_peaks_assoc["chrom"] == "chr11")]

,hm,lnc,lnc_ens,peak,chrom,start,end,corr,gene_chrom,gene_start,gene_end,gene_ens
106296720,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74289292.0,74528886.0,CATG00000002919-gene
106296721,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74329976.0,74398392.0,ENSG00000165434-gene
106296722,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74351444.0,74485742.0,ENSG00000254631-gene
106296723,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74366871.0,74511511.0,ENSG00000254837-gene
106296724,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74390265.0,74394560.0,CATG00000006156-gene
106296725,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74397766.0,74401595.0,ENSG00000264402-gene


to_file = f"{ID_HISTORY_CONVERTER_RES}/bad_genes-resolutions.tsv"
df_bad_genes_resolutions_final = pd.read_csv(to_file, sep="\t")

dict_bad_genes_resolutions_final = {i:j for i,j in zip(df_bad_genes_resolutions_final["name"], df_bad_genes_resolutions_final["name_latest"])}
len(dict_bad_genes_resolutions_final)

dict_bad_genes_resolutions_final

lnc_ens_list = himorna_peaks_assoc["lnc_ens"].unique()
len(lnc_ens_list)

len((set(lnc_ens_list) & set(list(dict_bad_genes_resolutions_final.keys()))))# лучше просто перенести их на последнюю аннотацию генкода. Также как в статье Гришы Рябухина, черещ индекс Жаккара

himorna_peaks_assoc["gene_ens"] = himorna_peaks_assoc["gene_ens"].astype(str)

himorna_peaks_assoc["gene_ens-updated"] = himorna_peaks_assoc["gene_ens"].apply(lambda x: dict_bad_genes_resolutions_final[x.split("-")[0]]+"-gene" if x.split("-")[0] in dict_bad_genes_resolutions_final else x)

In [120]:
himorna_peaks_assoc.groupby(by="hm").agg({"lnc_ens":pd.Series.nunique, "gene_ens":pd.Series.nunique})

,lnc_ens,gene_ens
hm,,
H3K27ac,2031,107977
H3K27me3,1401,90337
H3K36me3,1321,100325
H3K4me1,1954,113410
H3K4me2,1031,87055
H3K4me3,1752,96787
H3K79me2,802,63039
H3K9ac,1104,44435
H3K9me3,1079,79152
